# Warm-up: a plain network on a simulated panel

Work package 2, first step: a neural network (no hybrid model yet) learns the pharmacokinetics and pharmacodynamics of a panel of virtual patients, which the mechanistic apixaban model simulates.

The code is the package `hybrid` of the repository; this notebook installs it, runs it and shows the results. Generated files (dataset, weights, logs, figures) are written to Google Drive, not to the repository.

## Setup

Set the repository, the branch or commit, and the folder of Google Drive for the generated files.

In [ ]:
REPO = "https://github.com/matthiaskoenig/neural-ode.git"
REF = "main"  # branch, tag or commit
RUNS = "/content/drive/MyDrive/hybrid_runs"  # generated files; "MyDrive" is "Meine Ablage"

In [ ]:
import os
import sys

from google.colab import drive

drive.mount("/content/drive")
os.makedirs(RUNS, exist_ok=True)
os.environ["HYBRID_RUNS"] = RUNS

!rm -rf /content/neural-ode
!git clone -q {REPO} /content/neural-ode && git -C /content/neural-ode checkout -q {REF}
!pip install -q -e /content/neural-ode/sbml2jax -e /content/neural-ode/hybrid

# the packages of the clone, for the cells of this notebook
sys.path[:0] = ["/content/neural-ode/hybrid/src", "/content/neural-ode/sbml2jax/src"]
CONFIGS = "/content/neural-ode/hybrid/configs"
!git -C /content/neural-ode log -1 --format="commit %h: %s"

## Train

`hybrid train` simulates the panel (once, then it is read from Drive), trains the network with Adam and L-BFGS, and writes the run: `config.toml`, `run.json` (commit, seed, versions), `weights.eqx`, `log.csv`, `metrics.json` and the figures.

In [ ]:
!hybrid train {CONFIGS}/warmup_apixaban.toml

## Results

The newest run of the case: validation patients, which the training has not seen.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

run = sorted((Path(RUNS) / "warmup_apixaban").iterdir())[-1]
print(run)
for name in ("fits.png", "parity.png", "learning.png"):
    display(Image(filename=str(run / name), width=900))

## Another case

A case is one TOML file: model, dose, covariates, observables with their noise, network and training. Copy the file to Drive, edit it there (double click in the file browser on the left) and train it. A change of the data settings generates a new dataset; a change of the network or the training reuses the dataset.

In [ ]:
import shutil

case = Path(RUNS) / "cases" / "my_case.toml"
case.parent.mkdir(exist_ok=True)
if not case.exists():
    shutil.copyfile(f"{CONFIGS}/warmup_apixaban.toml", case)
    # every case has its own name, the directory of its runs
    case.write_text(case.read_text().replace('name = "warmup_apixaban"', 'name = "my_case"', 1))
print(case)

In [ ]:
!hybrid train "{case}"

## A run in Python

`warmup.load` gives the configuration, the panel and the trained model of a run, to try things out on top of the package: other figures, other inputs, a network of your own with `hybrid.training.fit.fit`.

In [ ]:
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from hybrid import warmup

config, panel, model = warmup.load(run)
print(panel.x_names, panel.y_names, panel.y.shape)

# the first observable for the dose levels of a typical patient
times = jnp.linspace(panel.times[0], panel.times[-1], 200)
typical = np.median(panel.x, axis=0)
for dose in np.unique(panel.x[:, 0]):
    x = jnp.asarray([[dose, *typical[1:]]])
    plt.plot(times, model.network.predict(times, x)[0, :, 0], label=f"{panel.x_names[0]} {dose:g}")
plt.xlabel("time")
plt.ylabel(panel.y_names[0])
plt.legend()
plt.show()